In [3]:
# Cell 1 — Install dependencies
!pip install -q librosa soundfile numpy scikit-learn

In [4]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Cell 3 — Imports
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import pickle

print('✅ Imports done!')

✅ Imports done!


In [6]:
# Cell 4 — Define all paths
BASE      = '/content/drive/MyDrive/Hackenza_MUPlovers/'
AUDIO_DIR = BASE + 'data/processed/'
CHUNK_CSV = BASE + 'data/chunk_index.csv'

# Input dirs
EMB_DIR   = BASE + 'cache/embeddings_base/'         # mixed old + new embeddings
PROS_DIR  = BASE + 'cache/features/prosody/'   # prosody features [T, 10]
NOISE_DIR = BASE + 'cache/noise/'              # noise features [T, 5]

# Separated embedding dirs
EMB_768_DIR  = BASE + 'cache/embeddings_768/'   # WavLM-Base  [T, 768]
EMB_1024_DIR = BASE + 'cache/embeddings_1024/'  # WavLM-Large [T, 1024]

# Assembly output dirs
FEAT_783_DIR  = BASE + 'cache/features_783/'    # old [T, 783]
FEAT_1039_DIR = BASE + 'cache/features_1039/'   # new [T, 1039]

# Final normalized output — same location as before
NORM_DIR = BASE + 'cache/features_normalized/'

# Create all folders
for d in [EMB_768_DIR, EMB_1024_DIR, NOISE_DIR,
          FEAT_783_DIR, FEAT_1039_DIR, NORM_DIR]:
    os.makedirs(d, exist_ok=True)

# Load chunk index
chunk_index = pd.read_csv(CHUNK_CSV)

print('BASE exists    :', os.path.exists(BASE))
print('AUDIO_DIR exists:', os.path.exists(AUDIO_DIR))
print('EMB_DIR exists :', os.path.exists(EMB_DIR))
print('PROS_DIR exists:', os.path.exists(PROS_DIR))
print('Chunk index rows:', len(chunk_index))
print('Unique files    :', chunk_index['file_id'].nunique())
print('\n✅ Paths defined!')

BASE exists    : True
AUDIO_DIR exists: True
EMB_DIR exists : True
PROS_DIR exists: True
Chunk index rows: 5179
Unique files    : 160

✅ Paths defined!


In [7]:
# Cell 5 — Sort mixed embeddings into separate folders by dim
print('Sorting embeddings by dimension...')
print('Checking', len(os.listdir(EMB_DIR)), 'files in embeddings/')

moved_768  = 0
moved_1024 = 0
skipped    = 0

for fname in tqdm(sorted(os.listdir(EMB_DIR))):
    if not fname.endswith('.npy'):
        continue

    fpath = os.path.join(EMB_DIR, fname)
    emb   = np.load(fpath)

    if emb.shape[1] == 768:
        np.save(os.path.join(EMB_768_DIR, fname), emb)
        moved_768 += 1

    elif emb.shape[1] == 1024:
        np.save(os.path.join(EMB_1024_DIR, fname), emb)
        moved_1024 += 1

    else:
        print(f'⚠️  Unknown dim {emb.shape} for {fname}')
        skipped += 1

print(f'\n✅ Sorting done!')
print(f'   WavLM-Base  (768)  → embeddings_768/  : {moved_768} files')
print(f'   WavLM-Large (1024) → embeddings_1024/ : {moved_1024} files')
if skipped:
    print(f'   ⚠️ Skipped unknown dims: {skipped} files')

Sorting embeddings by dimension...
Checking 160 files in embeddings/


100%|██████████| 160/160 [00:54<00:00,  2.95it/s]


✅ Sorting done!
   WavLM-Base  (768)  → embeddings_768/  : 160 files
   WavLM-Large (1024) → embeddings_1024/ : 0 files


In [8]:
# Cell 6 — Define noise feature extraction
def extract_noise_features(audio_chunk, sr=16000):
    """
    Extracts 5 noise features from a single audio chunk.
    Returns numpy array of shape [5]
    """
    if len(audio_chunk) < 512 or np.max(np.abs(audio_chunk)) < 1e-6:
        return np.zeros(5, dtype=np.float32)

    stft           = np.abs(librosa.stft(audio_chunk, n_fft=512, hop_length=256))
    power_spectrum = stft ** 2

    # Feature 1 — SNR Proxy
    rms          = librosa.feature.rms(y=audio_chunk, frame_length=512, hop_length=256)[0]
    sorted_rms   = np.sort(rms)
    n            = len(sorted_rms)
    noise_floor  = np.mean(sorted_rms[:max(1, n//10)])
    signal_level = np.mean(sorted_rms[max(1, n*9//10):])
    snr_proxy    = float(np.log1p(signal_level / (noise_floor + 1e-8)))

    # Feature 2 — Spectral Entropy
    power_norm        = power_spectrum / (power_spectrum.sum(axis=0, keepdims=True) + 1e-8)
    entropy_per_frame = -np.sum(power_norm * np.log(power_norm + 1e-8), axis=0)
    spectral_entropy  = float(np.mean(entropy_per_frame))

    # Feature 3 — Spectral Flatness
    flatness          = librosa.feature.spectral_flatness(y=audio_chunk, n_fft=512, hop_length=256)[0]
    spectral_flatness = float(np.mean(flatness))

    # Feature 4 — Spectral Centroid
    centroid          = librosa.feature.spectral_centroid(y=audio_chunk, sr=sr, n_fft=512, hop_length=256)[0]
    spectral_centroid = float(np.mean(centroid))

    # Feature 5 — Spectral Bandwidth
    bandwidth          = librosa.feature.spectral_bandwidth(y=audio_chunk, sr=sr, n_fft=512, hop_length=256)[0]
    spectral_bandwidth = float(np.mean(bandwidth))

    return np.array([
        snr_proxy, spectral_entropy, spectral_flatness,
        spectral_centroid, spectral_bandwidth
    ], dtype=np.float32)

print('✅ Noise function defined!')

✅ Noise function defined!


In [9]:
# Cell 7 — Extract noise features for all files
all_file_ids = chunk_index['file_id'].unique()
failed       = []

print(f'Extracting noise for {len(all_file_ids)} files...')

for file_id in tqdm(all_file_ids):
    save_path = os.path.join(NOISE_DIR, f'{file_id}.npy')

    if os.path.exists(save_path):
        continue

    try:
        file_chunks      = chunk_index[chunk_index['file_id'] == file_id]
        chunk_boundaries = list(zip(file_chunks['start_sample'], file_chunks['end_sample']))
        audio, _         = librosa.load(AUDIO_DIR + f'{file_id}.wav', sr=16000, mono=True)
        noise_matrix     = np.stack([
            extract_noise_features(audio[s:e]) for (s, e) in chunk_boundaries
        ], axis=0)
        np.save(save_path, noise_matrix)

    except Exception as e:
        print(f'❌ Failed {file_id}: {e}')
        failed.append(file_id)

print(f'\n✅ Noise done! {len(all_file_ids)-len(failed)}/{len(all_file_ids)} files')
if failed:
    print('❌ Failed:', failed)

Extracting noise for 160 files...


100%|██████████| 160/160 [00:00<00:00, 1424.82it/s]


✅ Noise done! 160/160 files


In [10]:
# Cell 8 — Define assembly function
def assemble_features(file_id, emb_dir, pros_dir, noise_dir, feat_dir):
    """
    Assembles embeddings + prosody + noise into one feature matrix.
    Saves to feat_dir and returns the matrix.
    """
    embeddings = np.load(os.path.join(emb_dir,   f'{file_id}.npy'))
    prosody    = np.load(os.path.join(pros_dir,   f'{file_id}.npy'))
    noise      = np.load(os.path.join(noise_dir,  f'{file_id}.npy'))

    T_emb  = embeddings.shape[0]
    T_pros = prosody.shape[0]
    T_noi  = noise.shape[0]

    if not (T_emb == T_pros == T_noi):
        raise ValueError(
            f'T mismatch for {file_id}! '
            f'emb={T_emb}, pros={T_pros}, noise={T_noi}'
        )

    X = np.concatenate([embeddings, prosody, noise], axis=1)
    os.makedirs(feat_dir, exist_ok=True)
    np.save(os.path.join(feat_dir, f'{file_id}.npy'), X)
    return X

print('✅ Assembly function defined!')

✅ Assembly function defined!


In [13]:
files_768 = [f.replace('.npy','') for f in os.listdir(EMB_768_DIR) if f.endswith('.npy')]
print(f'Assembling {len(files_768)} files with WavLM-Base embeddings (783-dim)...')
failed_783 = []

for file_id in tqdm(files_768):
    try:
        X = assemble_features(
            file_id   = file_id,
            emb_dir   = EMB_768_DIR,
            pros_dir  = PROS_DIR,
            noise_dir = NOISE_DIR,
            feat_dir  = FEAT_783_DIR
        )
    except Exception as e:
        print(f'❌ {file_id}: {e}')
        failed_783.append(file_id)

print(f'✅ 783-dim assembly done! {len(os.listdir(FEAT_783_DIR))} files')

Assembling 160 files with WavLM-Base embeddings (783-dim)...


100%|██████████| 160/160 [03:44<00:00,  1.40s/it]

✅ 783-dim assembly done! 160 files


In [14]:
USE_DIR = FEAT_783_DIR
USE_DIM = 783
print(f'Using 783-dim features (WavLM-Base) ✅')
print(f'Files to normalize: {len(os.listdir(USE_DIR))}')

Using 783-dim features (WavLM-Base) ✅
Files to normalize: 160


In [15]:
print(f'Loading all {USE_DIM}-dim features to fit scaler...')

all_file_ids_norm = [
    f.replace('.npy','')
    for f in os.listdir(USE_DIR)
    if f.endswith('.npy')
]

# Stack all chunks
all_chunks = []
for file_id in all_file_ids_norm:
    X = np.load(os.path.join(USE_DIR, f'{file_id}.npy'))
    all_chunks.append(X)

all_stacked = np.vstack(all_chunks)
print(f'Total chunks stacked: {all_stacked.shape}')

# Fit scaler
print('Fitting StandardScaler...')
scaler = StandardScaler()
scaler.fit(all_stacked)
print('✅ Scaler fitted!')

# Normalize and save
print('Normalizing and saving to features_normalized/...')
for file_id in tqdm(all_file_ids_norm):
    X      = np.load(os.path.join(USE_DIR, f'{file_id}.npy'))
    X_norm = scaler.transform(X).astype(np.float32)
    np.save(os.path.join(NORM_DIR, f'{file_id}.npy'), X_norm)

print(f'\n✅ Done! {len(os.listdir(NORM_DIR))} files saved to features_normalized/')

Loading all 783-dim features to fit scaler...
Total chunks stacked: (5179, 783)
Fitting StandardScaler...
✅ Scaler fitted!
Normalizing and saving to features_normalized/...


100%|██████████| 160/160 [00:52<00:00,  3.07it/s]


✅ Done! 160 files saved to features_normalized/


In [16]:
# Cell 12 — Normalize and save to features_normalized/
print(f'Loading all {USE_DIM}-dim features to fit scaler...')

all_file_ids_norm = [
    f.replace('.npy','')
    for f in os.listdir(USE_DIR)
    if f.endswith('.npy')
]

# Stack all chunks
all_chunks = []
for file_id in all_file_ids_norm:
    X = np.load(os.path.join(USE_DIR, f'{file_id}.npy'))
    all_chunks.append(X)

all_stacked = np.vstack(all_chunks)
print(f'Total chunks stacked: {all_stacked.shape}')

# Fit scaler
print('Fitting StandardScaler...')
scaler = StandardScaler()
scaler.fit(all_stacked)
print('✅ Scaler fitted!')

# Normalize and save
print('Normalizing and saving to features_normalized/...')
for file_id in tqdm(all_file_ids_norm):
    X      = np.load(os.path.join(USE_DIR, f'{file_id}.npy'))
    X_norm = scaler.transform(X).astype(np.float32)
    np.save(os.path.join(NORM_DIR, f'{file_id}.npy'), X_norm)

print(f'\n✅ Normalization done! {len(os.listdir(NORM_DIR))} files saved to features_normalized/')

Loading all 783-dim features to fit scaler...
Total chunks stacked: (5179, 783)
Fitting StandardScaler...
✅ Scaler fitted!
Normalizing and saving to features_normalized/...


100%|██████████| 160/160 [00:01<00:00, 93.53it/s]


✅ Normalization done! 160 files saved to features_normalized/


In [17]:
# Cell 13 — Save scaler
scaler_path = BASE + 'cache/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Scaler saved at:', scaler_path)

✅ Scaler saved at: /content/drive/MyDrive/Hackenza_MUPlovers/cache/scaler.pkl


In [18]:
# Cell 14 — Final verification
all_norm_files = [f for f in os.listdir(NORM_DIR) if f.endswith('.npy')]
shapes  = []
issues  = []

for fname in all_norm_files:
    X = np.load(os.path.join(NORM_DIR, fname))
    if X.shape[1] != USE_DIM:
        issues.append(f'WRONG D: {fname} → {X.shape}')
    else:
        shapes.append(X.shape[0])

# Check one file stats
sample = np.load(os.path.join(NORM_DIR, all_norm_files[0]))

print(f'✅ Total files       : {len(shapes)}/{len(all_norm_files)}')
print(f'✅ Feature dim D     : {USE_DIM}')
print(f'✅ T range           : min={min(shapes)}, max={max(shapes)}, avg={sum(shapes)//len(shapes)}')
print(f'✅ Mean (should be ~0): {sample.mean():.4f}')
print(f'✅ Std  (should be ~1): {sample.std():.4f}')

if issues:
    print('❌ Issues found:', issues)
else:
    print('\n✅ No issues! Ready to hand off to H!')
    print(f'→ Tell H to use input_dim={USE_DIM} in the model')

✅ Total files       : 160/160
✅ Feature dim D     : 783
✅ T range           : min=12, max=267, avg=32
✅ Mean (should be ~0): 0.0025
✅ Std  (should be ~1): 1.3086

✅ No issues! Ready to hand off to H!
→ Tell H to use input_dim=783 in the model
